# Financial News Based Market Impact Prediction using FinBERT

Predict whether a stock-related financial news headline is **Bullish**, **Bearish**, or **Neutral** using only the headline text. Labels are generated from next-trading-day stock returns, and models include TF-IDF baselines plus a fine-tuned FinBERT classifier.

### 1. Setup & Imports
First, let's load all the standard packages and define the mappings for labels.

In [ ]:
from __future__ import annotations
import os
import re
import sys
import time
import zipfile
from dataclasses import dataclass
from html import unescape
from pathlib import Path
from typing import Any
import pandas as pd
import numpy as np
import torch
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

# Global Label mappings
LABELS = ["Bearish", "Neutral", "Bullish"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

print("Setup complete. PyTorch GPU available:", torch.cuda.is_available())

### 2. Configuration
Define parameters, directories, and data processing settings.

In [ ]:
# Configuration dict
cfg = {
    "paths": {
        "raw_csv_path": "../analyst_ratings_processed_file.csv",
        "data_dir": "data",
        "raw_dir": "data/raw",
        "processed_dir": "data/processed",
        "artifacts_dir": "artifacts",
        "figures_dir": "reports/figures",
        "reports_dir": "reports",
    },
    "data": {
        "headline_col": "title",
        "date_col": "date",
        "ticker_col": "stock",
        "index_cols_to_drop": ["Unnamed: 0"],
        "min_ticker_rows": 2,
        "max_rows": None,  # Change this to an integer (e.g. 10000) for faster testing
        "random_state": 42,
        "validation_size": 0.15,
        "test_size": 0.15,
    },
    "labeling": {
        "bullish_threshold": 0.01,
        "bearish_threshold": -0.01,
        "yfinance_batch_size": 1,
        "price_start_buffer_days": 7,
        "price_end_buffer_days": 14,
        "max_tickers": None,  # Change this to an integer (e.g. 15) to sample quickly
        "sleep_seconds": 0.2,
    },
    "preprocessing": {
        "max_features": 50000,
        "ngram_range": [1, 2],
        "lowercase": True,
    },
    "baseline": {
        "models": ["logistic_regression", "random_forest", "xgboost"],
    },
    "finbert": {
        "model_name": "ProsusAI/finbert",
        "output_dir": "artifacts/finbert",
        "max_length": 96,
        "learning_rate": 0.00002,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "num_train_epochs": 3,
        "weight_decay": 0.01,
        "max_train_samples": None,
        "max_eval_samples": None,
    }
}

# Ensure output directories exist
for key in ["data_dir", "raw_dir", "processed_dir", "artifacts_dir", "figures_dir", "reports_dir"]:
    Path(cfg["paths"][key]).mkdir(parents=True, exist_ok=True)
print("Output directories verified.")

### 3. Data Cleaning & Text Preprocessing
Create helpers to clean text and load/format the dataset.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
HTML_TAG_RE = re.compile(r"<[^>]+>")
SPECIAL_RE = re.compile(r"[^a-z0-9$%&+\-.//s]")
SPACE_RE = re.compile(r"\s+")

def clean_headline(text: str) -> str:
    """Clean text while keeping finance-relevant symbols like $, %, +, -, and ticker dots."""
    if text is None:
        return ""
    text = HTML_TAG_RE.sub(" ", unescape(str(text)))
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = SPECIAL_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text

def load_raw_dataset(cfg: dict[str, Any]) -> pd.DataFrame:
    raw_path = Path(cfg["paths"]["raw_csv_path"])
    max_rows = cfg["data"].get("max_rows")
    df = pd.read_csv(raw_path, nrows=max_rows)
    drop_cols = [c for c in cfg["data"].get("index_cols_to_drop", []) if c in df.columns]
    return df.drop(columns=drop_cols)

def clean_raw_dataset(df: pd.DataFrame, cfg: dict[str, Any]) -> pd.DataFrame:
    headline_col = cfg["data"]["headline_col"]
    date_col = cfg["data"]["date_col"]
    ticker_col = cfg["data"]["ticker_col"]

    cleaned = df[[headline_col, date_col, ticker_col]].copy()
    cleaned = cleaned.rename(
        columns={headline_col: "headline", date_col: "date", ticker_col: "ticker"}
    )
    cleaned["headline"] = cleaned["headline"].astype("string").str.strip()
    cleaned["ticker"] = cleaned["ticker"].astype("string").str.upper().str.strip()
    cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce", utc=True)
    cleaned = cleaned.dropna(subset=["headline", "ticker", "date"])
    cleaned = cleaned[cleaned["headline"].str.len() > 0]
    cleaned = cleaned[cleaned["ticker"].str.len() > 0]
    return cleaned.reset_index(drop=True)

### 4. Exploratory Data Analysis (EDA)
Load dataset, inspect statistics, and generate standard EDA plots.

In [ ]:
print("Loading raw dataset...")
raw_df = load_raw_dataset(cfg)
print(f"Loaded {len(raw_df)} rows.")
cleaned_df = clean_raw_dataset(raw_df, cfg)
print(f"Cleaned dataset: {len(cleaned_df)} rows remaining.")

figures_dir = Path(cfg["paths"]["figures_dir"])
sns.set_theme(style="whitegrid", context="notebook")

# 1. Plot Missing Values
missing = raw_df.isna().sum().sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing.values, color="#2F80ED")
plt.title("Missing Values by Column")
plt.ylabel("Missing Count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(figures_dir / "missing_values.png", dpi=160)
plt.show()

# 2. Plot Ticker Distribution
counts = cleaned_df["ticker"].value_counts().head(25)
plt.figure(figsize=(11, 5))
sns.barplot(x=counts.index, y=counts.values, color="#27AE60")
plt.title("Top 25 Tickers by Headline Count")
plt.ylabel("Headlines")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(figures_dir / "ticker_distribution.png", dpi=160)
plt.show()

# 3. Plot Publication Date Distribution
by_month = cleaned_df.assign(month=cleaned_df["date"].dt.to_period("M").astype(str)).groupby("month").size()
plt.figure(figsize=(12, 5))
by_month.plot(color="#9B51E0")
plt.title("Publication Date Distribution by Month")
plt.ylabel("Headlines")
plt.xlabel("Month")
plt.tight_layout()
plt.savefig(figures_dir / "publication_distribution.png", dpi=160)
plt.show()

### 5. Label Generation via Stock Returns (`yfinance`)
Join the clean news headlines with historical asset prices to compute next-trading-day returns and generate labels.

In [ ]:
@dataclass(frozen=True)
class LabelThresholds:
    bullish: float = 0.03
    bearish: float = -0.03

def label_from_return(return_pct: float, thresholds: LabelThresholds) -> str:
    if return_pct > thresholds.bullish:
        return "Bullish"
    if return_pct < thresholds.bearish:
        return "Bearish"
    return "Neutral"

def _download_prices(ticker: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import yfinance as yf
    prices = yf.download(
        ticker,
        start=start.date().isoformat(),
        end=end.date().isoformat(),
        progress=False,
        auto_adjust=False,
        actions=False,
        threads=False,
    )
    if prices.empty:
        return pd.DataFrame(columns=["close"])
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)
    prices = prices.rename(columns={c: c.lower().replace(" ", "_") for c in prices.columns})
    close_col = "adj_close" if "adj_close" in prices.columns else "close"
    out = prices[[close_col]].rename(columns={close_col: "close"}).dropna()
    out.index = pd.to_datetime(out.index).tz_localize(None).normalize()
    return out

def _next_trading_return(prices: pd.DataFrame, event_date: pd.Timestamp) -> float | None:
    if prices.empty:
        return None
    
    # Convert publication timestamp to US/Eastern (market timezone)
    try:
        event_est = event_date.tz_convert("US/Eastern")
    except Exception:
        # Fallback if tz info is missing
        event_est = event_date.tz_localize("UTC").tz_convert("US/Eastern")
        
    # If published before 4:00 PM EST, it impacts today's session
    # If published after 4:00 PM EST, it impacts the next trading day's session
    if event_est.hour < 16:
        target_day = event_est.normalize().tz_localize(None)
    else:
        target_day = (event_est.normalize() + pd.Timedelta(days=1)).tz_localize(None)
        
    # Find trading days on or after the target day
    trading_days = prices.index[prices.index >= target_day]
    if len(trading_days) == 0:
        return None
    actual_target_day = trading_days[0]
    
    # Get the previous trading day's close price to compute the return
    prev_days = prices.index[prices.index < actual_target_day]
    if len(prev_days) == 0:
        return None
    prev_day = prev_days[-1]
    
    prev_close = float(prices.loc[prev_day, "close"])
    target_close = float(prices.loc[actual_target_day, "close"])
    if prev_close == 0:
        return None
    return (target_close - prev_close) / prev_close
def generate_market_impact_labels(df: pd.DataFrame, cfg: dict[str, Any]) -> pd.DataFrame:
    thresholds = LabelThresholds(
        bullish=cfg["labeling"]["bullish_threshold"],
        bearish=cfg["labeling"]["bearish_threshold"],
    )
    start_buffer = cfg["labeling"]["price_start_buffer_days"]
    end_buffer = cfg["labeling"]["price_end_buffer_days"]
    max_tickers = cfg["labeling"].get("max_tickers")
    sleep_seconds = cfg["labeling"].get("sleep_seconds", 0)

    # Select the top tickers with the most headlines
    tickers = df["ticker"].value_counts().index.tolist()
    if max_tickers:
        tickers = tickers[: int(max_tickers)]
        df = df[df["ticker"].isin(tickers)].copy()

    frames: list[pd.DataFrame] = []
    for ticker in tqdm(tickers, desc="Labeling tickers"):
        ticker_df = df[df["ticker"] == ticker].copy()
        start = ticker_df["date"].min() - pd.Timedelta(days=start_buffer)
        end = ticker_df["date"].max() + pd.Timedelta(days=end_buffer)
        try:
            prices = _download_prices(ticker, start, end)
            if prices.empty:
                continue
            ticker_df["return_pct"] = ticker_df["date"].apply(lambda d: _next_trading_return(prices, d))
            ticker_df = ticker_df.dropna(subset=["return_pct"])
            ticker_df["label"] = ticker_df["return_pct"].apply(lambda x: label_from_return(x, thresholds))
            frames.append(ticker_df[["headline", "ticker", "date", "return_pct", "label"]])
        except Exception as e:
            print(f"Skipping {ticker} due to: {e}")
        if sleep_seconds:
            time.sleep(float(sleep_seconds))

    if not frames:
        return pd.DataFrame(columns=["headline", "ticker", "date", "return_pct", "label"])
    return pd.concat(frames, ignore_index=True)

### 6. Create Dataset Splits & Label Distribution
Filter dataset and partition it into train/validation/test splits.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# ── PART 1: Load Twitter gold labels ──────────────────────────────
print("Loading Twitter Financial News Sentiment dataset...")
ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
twitter_train = ds["train"].to_pandas()
twitter_val   = ds["validation"].to_pandas()

label_map = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
twitter_train["label"] = twitter_train["label"].map(label_map)
twitter_val["label"]   = twitter_val["label"].map(label_map)
twitter_train = twitter_train.rename(columns={"text": "headline"})
twitter_val   = twitter_val.rename(columns={"text": "headline"})
twitter_train["source"] = "twitter"
twitter_val["source"]   = "twitter"

print(f"Twitter train: {len(twitter_train)}, val: {len(twitter_val)}")

# ── PART 2: Label your CSV dataset via yfinance ───────────────────
print("\nGenerating yfinance labels on CSV dataset...")

cfg["labeling"]["bullish_threshold"] =  0.02   # ±2% threshold
cfg["labeling"]["bearish_threshold"] = -0.02

raw_df_500k   = load_raw_dataset(cfg)
cleaned_500k  = clean_raw_dataset(raw_df_500k, cfg)
print(f"Cleaned CSV rows: {len(cleaned_500k)}")

labeled_csv = generate_market_impact_labels(cleaned_500k, cfg)
print(f"yfinance labeled rows: {len(labeled_csv)}")
labeled_csv["source"] = "yfinance"

# Rebalance yfinance labels
neutral_df = labeled_csv[labeled_csv["label"] == "Neutral"]
bullish_df = labeled_csv[labeled_csv["label"] == "Bullish"]
bearish_df = labeled_csv[labeled_csv["label"] == "Bearish"]
minority   = min(len(bullish_df), len(bearish_df))
neutral_sampled = neutral_df.sample(n=minority * 2, random_state=42)
labeled_csv = (
    pd.concat([neutral_sampled, bullish_df, bearish_df])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print("yfinance rebalanced distribution:")
print(labeled_csv["label"].value_counts())

# ── PART 3: Combine both datasets ────────────────────────────────
# Keep only columns both share
csv_subset     = labeled_csv[["headline", "label", "source"]]
twitter_subset = twitter_train[["headline", "label", "source"]]

combined = (
    pd.concat([twitter_subset, csv_subset])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
combined["clean_headline"] = combined["headline"]

print(f"\nCombined dataset size: {len(combined)}")
print("Combined distribution:")
print(combined["label"].value_counts())

# Plot
counts = combined["label"].value_counts()
plt.figure(figsize=(7, 4))
sns.barplot(x=counts.index, y=counts.values, palette=["#7F8C8D", "#27AE60", "#C0392B"])
plt.title("Combined Label Distribution (Twitter + yfinance)")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

# ── PART 4: Split ────────────────────────────────────────────────
# Twitter val becomes our val/test — it's clean human-labeled data
# so better to evaluate on it than on noisy yfinance labels
val_df, test_df = train_test_split(
    twitter_val,
    test_size=0.5,
    random_state=42,
    stratify=twitter_val["label"]
)
val_df["clean_headline"]  = val_df["headline"]
test_df["clean_headline"] = test_df["headline"]

train_df = combined

processed_dir = Path(cfg["paths"]["processed_dir"])
train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir   / "validation.csv", index=False)
test_df.to_csv(processed_dir  / "test.csv", index=False)

print(f"\nTrain size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")


### 7. TF-IDF Baseline Models
Train baseline classifiers (Logistic Regression, Random Forest, XGBoost) and save metrics.

In [ ]:
def evaluate_predictions(y_true, y_pred, labels: list[str]) -> dict:
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "classification_report": report,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }

def build_baseline_model(name: str, cfg: dict[str, Any]) -> Pipeline:
    vectorizer = TfidfVectorizer(
        preprocessor=clean_headline,
        max_features=cfg["preprocessing"]["max_features"],
        ngram_range=tuple(cfg["preprocessing"]["ngram_range"]),
        min_df=2,
        sublinear_tf=True,
    )
    if name == "logistic_regression":
        clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=1)
    elif name == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=cfg["data"]["random_state"],
            n_jobs=1,
        )
    elif name == "xgboost":
        from xgboost import XGBClassifier
        clf = XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=cfg["data"]["random_state"],
            n_jobs=1,
        )
    else:
        raise ValueError(f"Unknown baseline model: {name}")
    return Pipeline([("tfidf", vectorizer), ("clf", clf)])

baseline_results = []
for model_name in cfg["baseline"]["models"]:
    print(f"Training baseline model: {model_name}...")
    try:
        model = build_baseline_model(model_name, cfg)
        y_train = train_df["label"]
        if model_name == "xgboost":
            label_to_id = {label: idx for idx, label in enumerate(LABELS)}
            model.fit(train_df["clean_headline"], y_train.map(label_to_id))
            pred_ids = model.predict(test_df["clean_headline"])
            preds = [LABELS[int(i)] for i in pred_ids]
        else:
            model.fit(train_df["clean_headline"], y_train)
            preds = model.predict(test_df["clean_headline"])
        
        report = evaluate_predictions(test_df["label"], preds, LABELS)
        model_dir = Path(cfg["paths"]["artifacts_dir"]) / "baselines"
        model_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, model_dir / f"{model_name}.joblib")
        
        print(f"{model_name} accuracy: {report['accuracy']:.4f}, macro F1: {report['macro_f1']:.4f}")
        baseline_results.append({"model": model_name, **report})
    except Exception as e:
        print(f"Failed to train {model_name}: {e}")

### 8. FinBERT Fine-Tuning
Fine-tune the pretrained `ProsusAI/finbert` model on the generated labels.

In [ ]:
def _to_dataset(df: pd.DataFrame, tokenizer, max_length: int) -> Dataset:
    dataset = Dataset.from_pandas(
        pd.DataFrame(
            {
                "text": df["headline"].astype(str).tolist(),
                "label": df["label"].map(LABEL_TO_ID).astype(int).tolist(),
            }
        ),
        preserve_index=False,
    )
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)
    return dataset.map(tokenize, batched=True)

def compute_metrics(eval_pred) -> dict[str, float]:
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

print("Loading FinBERT and tokenizer...")
finbert_cfg = cfg["finbert"]
tokenizer = AutoTokenizer.from_pretrained(finbert_cfg["model_name"])
model = AutoModelForSequenceClassification.from_pretrained(
    finbert_cfg["model_name"],
    num_labels=3,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

train_ds = _to_dataset(train_df, tokenizer, finbert_cfg["max_length"])
val_ds = _to_dataset(val_df, tokenizer, finbert_cfg["max_length"])
test_ds = _to_dataset(test_df, tokenizer, finbert_cfg["max_length"])

# Compute dynamic class weights to handle imbalance
from torch import nn
total = len(train_df)
w_bearish = total / (3.0 * (train_df["label"] == "Bearish").sum())
w_neutral = total / (3.0 * (train_df["label"] == "Neutral").sum())
w_bullish = total / (3.0 * (train_df["label"] == "Bullish").sum())
class_weights = [w_bearish, w_neutral, w_bullish]
print("Calculated class weights:", class_weights)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is not None:
            weight = torch.tensor(self.class_weights, dtype=torch.float).to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

output_dir = Path(finbert_cfg["output_dir"])
args = TrainingArguments(
    output_dir=str(output_dir),
    learning_rate=finbert_cfg["learning_rate"],
    per_device_train_batch_size=finbert_cfg["train_batch_size"],
    per_device_eval_batch_size=finbert_cfg["eval_batch_size"],
    num_train_epochs=finbert_cfg["num_train_epochs"],
    weight_decay=finbert_cfg["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print("Starting FinBERT fine-tuning...")
trainer.train()

# Save model
trainer.save_model(output_dir / "best_model")
tokenizer.save_pretrained(output_dir / "best_model")
print("FinBERT model fine-tuned & saved successfully.")


### 9. Evaluation & Comparison
Evaluate all classifiers on the test dataset and generate comparison summaries.

In [ ]:
print("Evaluating FinBERT on test set...")
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
preds_labels = [ID_TO_LABEL[p] for p in preds]

finbert_report = evaluate_predictions(test_df["label"], preds_labels, LABELS)
pd.Series(finbert_report).to_json(Path(cfg["paths"]["reports_dir"]) / "finbert_evaluation.json", indent=2)

print(f"FinBERT test accuracy: {finbert_report['accuracy']:.4f}, macro F1: {finbert_report['macro_f1']:.4f}")

# Compare models
all_results = baseline_results + [{"model": "FinBERT", **finbert_report}]
comparison_df = pd.DataFrame(all_results)[["model", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]]
print("\nModel Comparison Table:")
print(comparison_df.to_markdown(index=False))

# Save comparison report
comparison_df.to_csv(Path(cfg["paths"]["reports_dir"]) / "model_comparison.csv", index=False)

### 10. Predict Market Impact (Inference Example)
Perform inference on a single financial headline.

In [ ]:
class FinBERTMarketImpactPredictor:
    def __init__(self, model_dir: str | Path):
        self.model_dir = Path(model_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()

    @torch.no_grad()
    def predict(self, headline: str) -> dict:
        # Tokenize headline directly without clean_headline to preserve pretrained features
        inputs = self.tokenizer(headline, return_tensors="pt", truncation=True, max_length=96)
        logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
        id2label = self.model.config.id2label
        probabilities = {id2label[i]: float(probs[i]) for i in range(len(probs))}
        predicted_class = max(probabilities, key=probabilities.get)
        return {
            "headline": headline,
            "predicted_class": predicted_class,
            "confidence": probabilities[predicted_class],
            "probabilities": probabilities,
        }

# Test inference
predictor = FinBERTMarketImpactPredictor("artifacts/finbert/best_model")
result = predictor.predict("NVIDIA signs $50 billion AI infrastructure deal")
print("Result:", result)
